# SageMaker Processing Job — Sales Preprocessing (BYOC)

Preprocesamiento del dataset de ventas con un container propio (BYOC) que ejecuta
`preprocess.py` usando Polars + scikit-learn (IsolationForest).

### Flujo de datos

```
S3 (raw CSVs)  →  /opt/ml/processing/input/  →  preprocess.py  →  /opt/ml/processing/output/  →  S3 (parquets)
```

### Outputs generados

| Subdirectorio | Archivo | Descripción |
|---|---|---|
| `augmented/` | `augmented_sales.parquet` | Ventas con todas las features |
| `cleaned/` | `augmented_sales_cleaned.parquet` | Sin outliers |
| `monthly/` | `monthly_sales.parquet` | Agregación mensual + lags + rolling |
| `inference/` | `shops_catalog.parquet` | Catálogo de tiendas enriquecido |
| `inference/` | `items_catalog.parquet` | Catálogo de items con antigüedad |
| `inference/` | `items_categories_catalog.parquet` | Catálogo de categorías |

## 1. Setup

In [1]:
from time import gmtime, strftime
import boto3
import sagemaker
import os
from pathlib import Path

sagemaker_session = sagemaker.Session()
role              = sagemaker.get_execution_role()
bucket            = 'sagemaker-demand-retail'
region            = sagemaker_session.boto_region_name
timestamp         = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

# Prefijos S3
prefix            = "byoc-v1/sales-processing"
input_prefix      = f"{prefix}/input/raw"
output_prefix     = f"{prefix}/output/{timestamp}"

print(f"Role   : {role}")
print(f"Bucket : {bucket}")
print(f"Region : {region}")
print(f"Input  : s3://{bucket}/{input_prefix}/")
print(f"Output : s3://{bucket}/{output_prefix}/")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Role   : arn:aws:iam::433956820129:role/SageMakerStudioExecutionRole2026
Bucket : sagemaker-demand-retail
Region : us-east-1
Input  : s3://sagemaker-demand-retail/byoc-v1/sales-processing/input/raw/
Output : s3://sagemaker-demand-retail/byoc-v1/sales-processing/output/2026-04-07-00-58-08/


## 2. Carga del dataset a S3

Sube los 4 archivos CSV crudos al bucket de SageMaker.

In [2]:
BASE_DIR = Path(os.getcwd()).parent.parent
RAW_DIR = BASE_DIR / "data" / "raw"
print(f"Directorio de trabajo: {RAW_DIR}")

Directorio de trabajo: /home/sagemaker-user/Demand_Retail/data/raw


In [3]:
raw_files = [
    "sales_train.csv",
    "items_en.csv",
    "item_categories_en.csv",
    "shops_en.csv",
]

for f in raw_files:
    full_path = os.path.join(RAW_DIR, f)
    
    if not os.path.exists(full_path):
        print(f"Archivo no encontrado en {full_path}")
    else:
        s3_uri = sagemaker_session.upload_data(
            path=full_path,
            bucket=bucket,
            key_prefix=input_prefix,
        )
        print(f"  ✓ {f}  →  {s3_uri}")

  ✓ sales_train.csv  →  s3://sagemaker-demand-retail/byoc-v1/sales-processing/input/raw/sales_train.csv
  ✓ items_en.csv  →  s3://sagemaker-demand-retail/byoc-v1/sales-processing/input/raw/items_en.csv
  ✓ item_categories_en.csv  →  s3://sagemaker-demand-retail/byoc-v1/sales-processing/input/raw/item_categories_en.csv
  ✓ shops_en.csv  →  s3://sagemaker-demand-retail/byoc-v1/sales-processing/input/raw/shops_en.csv


## 3. Configuración del container BYOC

Construye la imagen Docker y la publica en Amazon ECR.

In [4]:
account_id = boto3.client("sts").get_caller_identity()["Account"]
ecr_repo   = "demand-retail-preprocessing"
tag        = "latest"
image_uri  = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo}:{tag}"

print(f"Image URI: {image_uri}")

Image URI: 433956820129.dkr.ecr.us-east-1.amazonaws.com/demand-retail-preprocessing:latest


In [6]:
# Construir imagen (ejecuta desde el directorio processing/container/)
%cd container
!docker build --network sagemaker -t {ecr_repo}:{tag} .

/home/sagemaker-user/Demand_Retail/src/preprocessing/container
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  23.04kB
Step 1/4 : FROM python:3.11-slim
3.11-slim: Pulling from library/python

1dee3f47: Pulling fs layer 
007a5b45: Pulling fs layer 
92e2619d: Pulling fs layer 
Digest: sha256:9358444059ed78e2975ada2c189f1c1a3144a5dab6f35bff8c981afb38946634
Status: Downloaded newer image for python:3.11-slim
 ---> e67db9b14d09
Step 2/4 : WORKDIR /opt/ml/processing
 ---> Running in ff95a2bb1fa8
 ---> Removed intermediate container ff95a2bb1fa8
 ---> 283ae9a2fb07
Step 3/4 : RUN pip install --no-cache-dir     numpy==2.2.3     polars==1.27.1     scikit-learn==1.6.1     pyarrow==19.0.1
 ---> Running in 6b7617d0221f
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.8 MB/s eta 0:00:00
   

In [7]:
# Login en ECR, crear repositorio y hacer push
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!aws ecr create-repository --repository-name {ecr_repo} --region {region} || true
!docker tag {ecr_repo}:{tag} {image_uri}
!docker push {image_uri}

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-east-1:433956820129:repository/demand-retail-preprocessing",
        "registryId": "433956820129",
        "repositoryName": "demand-retail-preprocessing",
        "repositoryUri": "433956820129.dkr.ecr.us-east-1.amazonaws.com/demand-retail-preprocessing",
        "createdAt": "2026-04-07T01:08:19.773000+00:00",
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": false
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [433956820129.dkr.ecr.us-east-1.amazonaws.com/demand-retail-preprocessing]

45fa32ef: Preparing 
1c4ddd1b: Preparing 
a0657a00:

## 4. Ejecución del Processing Job

`ScriptProcessor` inyecta `preprocess.py` en el container y lo ejecuta con `python3`.

| Parámetro | Valor | Descripción |
|---|---|---|
| `instance_type` | `ml.t3.large` | Es el tipo de instancia más grande que permite la cuota |
| `instance_count` | `1` | Una sola instancia |
| `command` | `["python3"]` | SageMaker ejecuta `python3 preprocess.py [args]` |

In [14]:
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor

processor = ScriptProcessor(
    base_job_name  = "sales-preprocessing",
    image_uri      = image_uri,
    command        = ["python3"],
    role           = role,
    instance_count = 1,
    instance_type  = "ml.t3.large",
    max_runtime_in_seconds = 3600,
)

processor.run(
    code      = "preprocess.py",
    arguments = [
        "--sales-file",    "sales_train.csv",
        "--contamination", "0.02",
        "--random-state",  "42",
    ],
    inputs = [
        ProcessingInput(
            source      = f"s3://{bucket}/{input_prefix}/",
            destination = "/opt/ml/processing/input",
        )
    ],
    outputs = [
        ProcessingOutput(
            output_name = "augmented",
            source      = "/opt/ml/processing/output/augmented",
            destination = f"s3://{bucket}/{output_prefix}/augmented",
        ),
        ProcessingOutput(
            output_name = "cleaned",
            source      = "/opt/ml/processing/output/cleaned",
            destination = f"s3://{bucket}/{output_prefix}/cleaned",
        ),
        ProcessingOutput(
            output_name = "monthly",
            source      = "/opt/ml/processing/output/monthly",
            destination = f"s3://{bucket}/{output_prefix}/monthly",
        ),
        ProcessingOutput(
            output_name = "inference",
            source      = "/opt/ml/processing/output/inference",
            destination = f"s3://{bucket}/{output_prefix}/inference",
        ),
    ],
    logs = True,
)

INFO:sagemaker:Creating processing-job with name sales-preprocessing-2026-04-07-01-21-58-427


........2026-04-07 01:23:17 [INFO] === SageMaker Processing Job — Sales Preprocessing ===
2026-04-07 01:23:17 [INFO] Input dir  : /opt/ml/processing/input
2026-04-07 01:23:17 [INFO] Output dir : /opt/ml/processing/output
2026-04-07 01:23:17 [INFO] Sales file : sales_train.csv
2026-04-07 01:23:17 [INFO] Contamination: 0.02 | Random state: 42
2026-04-07 01:23:17 [INFO] Cargando datos...
2026-04-07 01:23:18 [INFO]   2,935,849 registros cargados
2026-04-07 01:23:18 [INFO] Creando features...
2026-04-07 01:23:20 [INFO]   augmented_data shape: (2935849, 15)
2026-04-07 01:23:20 [INFO] Detectando outliers...
2026-04-07 01:24:02 [INFO]   57,104 registros eliminados como outliers
2026-04-07 01:24:02 [INFO] Agregando datos mensualmente...
2026-04-07 01:24:13 [INFO]   1,592,543 registros mensuales generados
2026-04-07 01:24:13 [INFO] Exportando outputs...
2026-04-07 01:24:15 [INFO]   ✓ augmented/augmented_sales.parquet
2026-04-07 01:24:16 [INFO]   ✓ cleaned/augmented_sales_cleaned.parquet
2026-04-

## 5. Inspección del output

Lee los primeros registros del parquet mensual desde S3 para validar el preprocessing.

In [15]:
import pandas as pd

# Nombre del job recién ejecutado
job_name = processor.latest_job.name
print(f"Job name: {job_name}")

Job name: sales-preprocessing-2026-04-07-01-21-58-427


In [17]:
# Descarga y muestra las primeras filas de monthly_sales.parquet
monthly_s3_path = f"s3://{bucket}/{output_prefix}/monthly/monthly_sales.parquet"
print(f"Leyendo: {monthly_s3_path}")

!aws s3 cp {monthly_s3_path} /tmp/monthly_sales.parquet

df_monthly = pd.read_parquet("/tmp/monthly_sales.parquet")
print(f"\nShape: {df_monthly.shape}")
print(f"Columnas: {df_monthly.columns}\n")
print(df_monthly.head(10))

Leyendo: s3://sagemaker-demand-retail/byoc-v1/sales-processing/output/2026-04-07-00-58-08/monthly/monthly_sales.parquet
download: s3://sagemaker-demand-retail/byoc-v1/sales-processing/output/2026-04-07-00-58-08/monthly/monthly_sales.parquet to ../../../../../../tmp/monthly_sales.parquet

Shape: (1592543, 21)
Columnas: Index(['year_month_date', 'month', 'shop_id', 'segment', 'macro_region',
       'item_id', 'general_category', 'device', 'item_age', 'avg_price',
       'monthly_items', 'monthly_items_lag_1', 'monthly_items_lag_12',
       'rolling_mean_3', 'rolling_mean_6', 'rolling_std_3', 'rolling_std_6',
       'trend_3', 'group_avg_region_category', 'group_avg_category_age',
       'group_avg_segment_age'],
      dtype='object')

  year_month_date  month shop_id segment macro_region item_id  \
0      2013-01-01      1       0     Low      Siberia    1000   
1      2013-02-01      2       0     Low      Siberia    1000   
2      2013-02-01      2       0     Low      Siberia   10004 

In [18]:
# Verificar los 6 archivos de salida en S3
print(f"Archivos en s3://{bucket}/{output_prefix}/\n")
!aws s3 ls --recursive s3://{bucket}/{output_prefix}/

Archivos en s3://sagemaker-demand-retail/byoc-v1/sales-processing/output/2026-04-07-00-58-08/

2026-04-07 01:24:33   13997622 byoc-v1/sales-processing/output/2026-04-07-00-58-08/augmented/augmented_sales.parquet
2026-04-07 01:24:33   12606585 byoc-v1/sales-processing/output/2026-04-07-00-58-08/cleaned/augmented_sales_cleaned.parquet
2026-04-07 01:24:33     466507 byoc-v1/sales-processing/output/2026-04-07-00-58-08/inference/items_catalog.parquet
2026-04-07 01:24:33       3690 byoc-v1/sales-processing/output/2026-04-07-00-58-08/inference/items_categories_catalog.parquet
2026-04-07 01:24:33       3867 byoc-v1/sales-processing/output/2026-04-07-00-58-08/inference/shops_catalog.parquet
2026-04-07 01:24:33   11267912 byoc-v1/sales-processing/output/2026-04-07-00-58-08/monthly/monthly_sales.parquet


In [19]:
# Estadísticas básicas del dataset mensual
print("=== Estadísticas básicas — monthly_sales ===")
print(df_monthly.describe())

=== Estadísticas básicas — monthly_sales ===
              month     avg_price  monthly_items  monthly_items_lag_1  \
count  1.592543e+06  1.592543e+06   1.592543e+06         1.592543e+06   
mean   6.152250e+00  7.571024e+02   2.058969e+00         1.717394e+00   
std    3.454502e+00  1.458482e+03   4.097489e+00         4.018947e+00   
min    1.000000e+00  9.000000e-02   0.000000e+00        -2.200000e+01   
25%    3.000000e+00  1.990000e+02   1.000000e+00         0.000000e+00   
50%    6.000000e+00  3.923300e+02   1.000000e+00         1.000000e+00   
75%    9.000000e+00  8.490000e+02   2.000000e+00         2.000000e+00   
max    1.200000e+01  3.449000e+04   6.470000e+02         5.350000e+02   

       monthly_items_lag_12  rolling_mean_3  rolling_mean_6  rolling_std_3  \
count          1.592543e+06    1.592543e+06    1.592543e+06   1.592543e+06   
mean           3.556909e-01    1.139594e+00    6.838611e-01   5.053641e-01   
std            2.880968e+00    3.507158e+00    3.088502e+00   1

---
## Resumen

| Output | Descripción |
|---|---|
| `augmented_sales.parquet` | Dataset enriquecido con features de tiendas, items y categorías |
| `augmented_sales_cleaned.parquet` | Dataset anterior sin outliers (IsolationForest, contamination=0.02) |
| `monthly_sales.parquet` | Agregación mensual por shop-item con lags, rolling means y trend |
| `shops_catalog.parquet` | macro_region, segment por tienda |
| `items_catalog.parquet` | item_age por producto |
| `items_categories_catalog.parquet` | general_category, device por categoría |